In [ ]:
import os, cv2, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

In [ ]:
I was doing google colab , it give Limited GPU , So that ,I have to Kaggle Nootbook .So that in Google colab Build model , There i was Dowload the Model there so that ,I can buld model forthere with the Retinal Oct dataset  

In [ ]:
import gdown
file_id = "1pX0eATFW5Wct0-j1cwwulLxA9yBDWLKh"   # your extracted ID
!gdown https://drive.google.com/uc?id={file_id} -O /kaggle/working/final_model.pth

In [ ]:
import os
oct_root = None
for dataset in os.listdir("/kaggle/input"):
    dataset_path = os.path.join("/kaggle/input", dataset)
    if not os.path.isdir(dataset_path): continue
    for root, dirs, files in os.walk(dataset_path):
        if 'train' in dirs and 'val' in dirs and 'test' in dirs:
            train_p = os.path.join(root, 'train')
            if all(c in os.listdir(train_p) for c in ['CNV','DME','DRUSEN','NORMAL']):
                oct_root = root
                break
    if oct_root: break
print("OCT root:", oct_root)

In [ ]:
class OCTDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root = os.path.join(root_dir, split)
        self.classes = sorted(os.listdir(self.root))
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
        self.samples = []
        for c in self.classes:
            class_dir = os.path.join(self.root, c)
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(('.jpeg','.jpg','.png')):
                    self.samples.append((os.path.join(class_dir, fname), self.class_to_idx[c]))
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = np.stack([img]*3, axis=-1)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(label, dtype=torch.long)

train_oct_ds = OCTDataset(oct_root, 'train', oct_train_transform)
val_oct_ds   = OCTDataset(oct_root, 'val',   oct_val_transform)
test_oct_ds  = OCTDataset(oct_root, 'test',  oct_val_transform)

oct_train_loader = DataLoader(train_oct_ds, batch_size=32, shuffle=True, num_workers=2)
oct_val_loader   = DataLoader(val_oct_ds,   batch_size=32, shuffle=False, num_workers=2)
oct_test_loader  = DataLoader(test_oct_ds,  batch_size=32, shuffle=False, num_workers=2)

print(f"OCT – Train: {len(train_oct_ds)}, Val: {len(val_oct_ds)}, Test: {len(test_oct_ds)}")

In [ ]:
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# ---------- set the exact base path you found ----------
dr_base = "/kaggle/input/datasets/mariaherrerot/aptos2019"

# Load the CSVs
train_df = pd.read_csv(os.path.join(dr_base, "train_1.csv"))
val_df   = pd.read_csv(os.path.join(dr_base, "valid.csv"))
test_df  = pd.read_csv(os.path.join(dr_base, "test.csv"))

# Helper: if a folder contains only one subfolder with the same name, go inside it
def get_img_dir(base, folder_name):
    p = os.path.join(base, folder_name)
    if os.path.isdir(p):
        items = os.listdir(p)
        if len(items) == 1 and os.path.isdir(os.path.join(p, items[0])):
            return os.path.join(p, items[0])
    return p

train_img_dir = get_img_dir(dr_base, "train_images")
val_img_dir   = get_img_dir(dr_base, "val_images")
test_img_dir  = get_img_dir(dr_base, "test_images")

print("Train images at:", train_img_dir)
print("Val   images at:", val_img_dir)
print("Test  images at:", test_img_dir)

# ---------- DR Dataset (same as before) ----------
class RetinalDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['id_code'] + '.png')
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(row['diagnosis'], dtype=torch.long)

# Create datasets with the transforms you already defined (dr_train_transform, dr_val_transform)
train_dr_ds = RetinalDataset(train_df, train_img_dir, dr_train_transform)
val_dr_ds   = RetinalDataset(val_df,   val_img_dir,   dr_val_transform)
test_dr_ds  = RetinalDataset(test_df,  test_img_dir,  dr_val_transform)

train_loader = DataLoader(train_dr_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dr_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dr_ds,  batch_size=32, shuffle=False, num_workers=2)

print(f"DR – Train: {len(train_dr_ds)}, Val: {len(val_dr_ds)}, Test: {len(test_dr_ds)}")

In [ ]:
# ---------- OCT transforms ----------
oct_train_transform = A.Compose([
    A.Resize(224,224), A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=0.5),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2), A.Rotate(limit=15, p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, p=0.3),
    A.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]), ToTensorV2()
])

oct_val_transform = A.Compose([
    A.Resize(224,224), A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=1.0),
    A.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]), ToTensorV2()
])

# ---------- DR (fundus) transforms (without GaussNoise) ----------
dr_train_transform = A.Compose([
    A.Resize(224,224), A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=0.5),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2), A.Rotate(limit=30, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    # A.GaussNoise(...) – removed because of albumentations version mismatch
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()
])

dr_val_transform = A.Compose([
    A.Resize(224,224), A.CLAHE(clip_limit=2.0, tile_grid_size=(8,8), p=1.0),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2()
])

In [ ]:
# Auto‑find the OCT dataset folder
oct_root = None
for dataset in os.listdir("/kaggle/input"):
    dataset_path = os.path.join("/kaggle/input", dataset)
    if not os.path.isdir(dataset_path): continue
    for root, dirs, files in os.walk(dataset_path):
        if 'train' in dirs and 'val' in dirs and 'test' in dirs:
            train_p = os.path.join(root, 'train')
            if all(c in os.listdir(train_p) for c in ['CNV','DME','DRUSEN','NORMAL']):
                oct_root = root
                break
    if oct_root: break

print("OCT root:", oct_root)

# OCT Dataset class
class OCTDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root = os.path.join(root_dir, split)
        self.classes = sorted(os.listdir(self.root))
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
        self.samples = []
        for c in self.classes:
            class_dir = os.path.join(self.root, c)
            for fname in os.listdir(class_dir):
                if fname.lower().endswith(('.jpeg','.jpg','.png')):
                    self.samples.append((os.path.join(class_dir, fname), self.class_to_idx[c]))
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = np.stack([img]*3, axis=-1)   # gray→3ch
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(label, dtype=torch.long)

# Instantiate datasets
train_oct_ds = OCTDataset(oct_root, 'train', oct_train_transform)
val_oct_ds   = OCTDataset(oct_root, 'val',   oct_val_transform)
test_oct_ds  = OCTDataset(oct_root, 'test',  oct_val_transform)

# DataLoaders
oct_train_loader = DataLoader(train_oct_ds, batch_size=32, shuffle=True, num_workers=2)
oct_val_loader   = DataLoader(val_oct_ds,   batch_size=32, shuffle=False, num_workers=2)
oct_test_loader  = DataLoader(test_oct_ds,  batch_size=32, shuffle=False, num_workers=2)

print(f"OCT – Train: {len(train_oct_ds)}, Val: {len(val_oct_ds)}, Test: {len(test_oct_ds)}")

In [ ]:
# DR base path – the folder that contains train_1.csv, valid.csv, etc.
dr_base = "/kaggle/input/datasets/mariaherrerot/aptos2019"

# Helper to handle nested folders (like train_images/train_images)
def get_img_dir(base, folder_name):
    p = os.path.join(base, folder_name)
    if os.path.isdir(p):
        items = os.listdir(p)
        if len(items) == 1 and os.path.isdir(os.path.join(p, items[0])):
            return os.path.join(p, items[0])
    return p

train_df = pd.read_csv(os.path.join(dr_base, "train_1.csv"))
val_df   = pd.read_csv(os.path.join(dr_base, "valid.csv"))
test_df  = pd.read_csv(os.path.join(dr_base, "test.csv"))

train_img_dir = get_img_dir(dr_base, "train_images")
val_img_dir   = get_img_dir(dr_base, "val_images")
test_img_dir  = get_img_dir(dr_base, "test_images")

print("DR train images at:", train_img_dir)

class RetinalDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['id_code'] + '.png')
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(row['diagnosis'], dtype=torch.long)

train_dr_ds = RetinalDataset(train_df, train_img_dir, dr_train_transform)
val_dr_ds   = RetinalDataset(val_df,   val_img_dir,   dr_val_transform)
test_dr_ds  = RetinalDataset(test_df,  test_img_dir,  dr_val_transform)

train_loader = DataLoader(train_dr_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dr_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dr_ds,  batch_size=32, shuffle=False, num_workers=2)

print(f"DR – Train: {len(train_dr_ds)}, Val: {len(val_dr_ds)}, Test: {len(test_dr_ds)}")

In [ ]:
class MultiHeadRetinalViT(nn.Module):
    def __init__(self, num_dr_classes=5, num_oct_classes=4, model_name='vit_base_patch16_224'):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0)
        self.embed_dim = self.backbone.embed_dim
        self.head_dr = nn.Linear(self.embed_dim, num_dr_classes)
        self.head_oct = nn.Linear(self.embed_dim, num_oct_classes)
    def forward(self, x, task='dr'):
        features = self.backbone(x)
        if task == 'dr': return self.head_dr(features)
        else: return self.head_oct(features)

import os
joint_ckpt = "/kaggle/working/multihead_final.pth"
dr_ckpt    = "/kaggle/working/final_model.pth"

if os.path.isfile(joint_ckpt):
    print("Loading joint model (already trained on both DR and OCT)")
    model = MultiHeadRetinalViT()
    model.load_state_dict(torch.load(joint_ckpt, map_location='cpu'))
elif os.path.isfile(dr_ckpt):
    print("Loading DR checkpoint only (OCT head will be random)")
    model = MultiHeadRetinalViT()
    ckpt = torch.load(dr_ckpt, map_location='cpu', weights_only=False)
    backbone_state = {k.replace('backbone.', ''): v for k,v in ckpt['model_state_dict'].items() if k.startswith('backbone.')}
    model.backbone.load_state_dict(backbone_state, strict=True)
    if 'head.weight' in ckpt['model_state_dict']:
        model.head_dr.load_state_dict({'weight': ckpt['model_state_dict']['head.weight'], 'bias': ckpt['model_state_dict']['head.bias']})
else:
    raise FileNotFoundError("No checkpoint found. Download final_model.pth first.")

model.to(device)
print("Model ready on", device)

In [ ]:
# Unfreeze last 4 transformer blocks
for blk in model.backbone.blocks[-4:]:
    for p in blk.parameters():
        p.requires_grad = True

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=0.01)
criterion_dr = nn.CrossEntropyLoss()
criterion_oct = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

epochs = 10
for epoch in range(1, epochs+1):
    model.train()
    total_dr, total_oct = 0.0, 0.0
    dr_iter = iter(train_loader)
    oct_iter = iter(oct_train_loader)
    n_batches = min(len(train_loader), len(oct_train_loader))

    for _ in range(n_batches):
        dr_img, dr_lbl = next(dr_iter)
        oct_img, oct_lbl = next(oct_iter)
        dr_img, dr_lbl = dr_img.to(device), dr_lbl.to(device)
        oct_img, oct_lbl = oct_img.to(device), oct_lbl.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            dr_out = model(dr_img, task='dr')
            dr_loss = criterion_dr(dr_out, dr_lbl)
            oct_out = model(oct_img, task='oct')
            oct_loss = criterion_oct(oct_out, oct_lbl)
            combined = dr_loss + oct_loss

        scaler.scale(combined).backward()
        scaler.step(optimizer)
        scaler.update()

        total_dr += dr_loss.item()
        total_oct += oct_loss.item()

    # Validation
    model.eval()
    correct_dr, correct_oct = 0, 0
    for img, lbl in val_loader:
        img, lbl = img.to(device), lbl.to(device)
        with torch.no_grad(), torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            correct_dr += (model(img, task='dr').argmax(1) == lbl).sum().item()
    for img, lbl in oct_val_loader:
        img, lbl = img.to(device), lbl.to(device)
        with torch.no_grad(), torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            correct_oct += (model(img, task='oct').argmax(1) == lbl).sum().item()

    dr_acc = correct_dr / len(val_loader.dataset)
    oct_acc = correct_oct / len(oct_val_loader.dataset)
    print(f"Epoch {epoch:2d} | DR loss {total_dr/n_batches:.4f} acc {dr_acc:.4f} | OCT loss {total_oct/n_batches:.4f} acc {oct_acc:.4f}")

    torch.save(model.state_dict(), f"/kaggle/working/joint_epoch{epoch}.pth")

In [12]:
import torch
from sklearn.metrics import classification_report

# Choose the best epoch
best_epoch = 4
ckpt_path = f"/kaggle/working/joint_epoch{best_epoch}.pth"
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

# --- DR Test Evaluation ---
correct_dr, all_preds_dr, all_labels_dr = 0, [], []
for img, lbl in test_loader:
    img, lbl = img.to(device), lbl.to(device)
    with torch.no_grad():
        out = model(img, task='dr')
        pred = out.argmax(1)
        correct_dr += (pred == lbl).sum().item()
        all_preds_dr.extend(pred.cpu().numpy())
        all_labels_dr.extend(lbl.cpu().numpy())
dr_acc = correct_dr / len(test_loader.dataset)
print(f"Epoch {best_epoch} DR Test Accuracy: {dr_acc:.4f}")
print(classification_report(all_labels_dr, all_preds_dr, target_names=['No DR','Mild','Moderate','Severe','Proliferative']))

# --- OCT Test Evaluation ---
correct_oct, all_preds_oct, all_labels_oct = 0, [], []
for img, lbl in oct_test_loader:
    img, lbl = img.to(device), lbl.to(device)
    with torch.no_grad():
        out = model(img, task='oct')
        pred = out.argmax(1)
        correct_oct += (pred == lbl).sum().item()
        all_preds_oct.extend(pred.cpu().numpy())
        all_labels_oct.extend(lbl.cpu().numpy())
oct_acc = correct_oct / len(oct_test_loader.dataset)
print(f"Epoch {best_epoch} OCT Test Accuracy: {oct_acc:.4f}")
print(classification_report(all_labels_oct, all_preds_oct, target_names=['CNV','DME','DRUSEN','NORMAL']))

Epoch 4 DR Test Accuracy: 0.8251
               precision    recall  f1-score   support

        No DR       0.96      0.99      0.97       199
         Mild       0.67      0.33      0.44        30
     Moderate       0.65      0.91      0.76        87
       Severe       0.40      0.24      0.30        17
Proliferative       0.86      0.36      0.51        33

     accuracy                           0.83       366
    macro avg       0.71      0.57      0.60       366
 weighted avg       0.83      0.83      0.81       366

Epoch 4 OCT Test Accuracy: 0.9866
              precision    recall  f1-score   support

         CNV       0.97      1.00      0.99       242
         DME       1.00      0.98      0.99       242
      DRUSEN       1.00      0.97      0.98       242
      NORMAL       0.98      1.00      0.99       242

    accuracy                           0.99       968
   macro avg       0.99      0.99      0.99       968
weighted avg       0.99      0.99      0.99       968



In [13]:
import shutil
shutil.copy(ckpt_path, "/kaggle/working/multihead_final.pth")

'/kaggle/working/multihead_final.pth'

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# ========================
# 1. Load best joint model (epoch 4)
# ========================
best_epoch = 4
ckpt_path = f"/kaggle/working/joint_epoch{best_epoch}.pth"
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.train()   # we will continue training

# ========================
# 2. Compute DR class weights (from training set)
# ========================
train_labels = train_df['diagnosis'].values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights (balanced):", class_weights)

# ========================
# 3. Define Focal Loss for DR and keep standard CE for OCT
# ========================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha.to(device) if alpha is not None else None
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)   # probability of correct class
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * (1 - pt) ** self.gamma * ce_loss
        else:
            focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

criterion_dr = FocalLoss(alpha=class_weights_tensor, gamma=2.0)
criterion_oct = nn.CrossEntropyLoss()   # OCT is already balanced

# ========================
# 4. Continue training (5 extra epochs)
# ========================
# Unfreeze last 4 blocks (keep them trainable if they were before; they already are)
for blk in model.backbone.blocks[-4:]:
    for p in blk.parameters():
        p.requires_grad = True

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

extra_epochs = 5
for epoch in range(1, extra_epochs+1):
    model.train()
    total_dr, total_oct = 0.0, 0.0
    dr_iter = iter(train_loader)
    oct_iter = iter(oct_train_loader)
    n_batches = min(len(train_loader), len(oct_train_loader))

    for _ in range(n_batches):
        dr_img, dr_lbl = next(dr_iter)
        oct_img, oct_lbl = next(oct_iter)
        dr_img, dr_lbl = dr_img.to(device), dr_lbl.to(device)
        oct_img, oct_lbl = oct_img.to(device), oct_lbl.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            dr_out = model(dr_img, task='dr')
            dr_loss = criterion_dr(dr_out, dr_lbl)
            oct_out = model(oct_img, task='oct')
            oct_loss = criterion_oct(oct_out, oct_lbl)
            combined = dr_loss + oct_loss

        scaler.scale(combined).backward()
        scaler.step(optimizer)
        scaler.update()

        total_dr += dr_loss.item()
        total_oct += oct_loss.item()

    # Validation
    model.eval()
    correct_dr, correct_oct = 0, 0
    for img, lbl in val_loader:
        img, lbl = img.to(device), lbl.to(device)
        with torch.no_grad(), torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            correct_dr += (model(img, task='dr').argmax(1) == lbl).sum().item()
    for img, lbl in oct_val_loader:
        img, lbl = img.to(device), lbl.to(device)
        with torch.no_grad(), torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            correct_oct += (model(img, task='oct').argmax(1) == lbl).sum().item()

    dr_acc = correct_dr / len(val_loader.dataset)
    oct_acc = correct_oct / len(oct_val_loader.dataset)
    print(f"Epoch {best_epoch+epoch:2d} | DR loss {total_dr/n_batches:.4f} acc {dr_acc:.4f} | OCT loss {total_oct/n_batches:.4f} acc {oct_acc:.4f}")

    torch.save(model.state_dict(), f"/kaggle/working/joint_epoch{best_epoch+epoch}.pth")

print("✅ Focal loss retraining complete.")

Class weights (balanced): [0.40864714 1.95333333 0.72524752 3.80519481 2.5042735 ]
Epoch  5 | DR loss 0.2806 acc 0.7514 | OCT loss 0.1672 acc 0.9688
Epoch  6 | DR loss 0.2262 acc 0.7541 | OCT loss 0.1732 acc 1.0000
Epoch  7 | DR loss 0.2224 acc 0.8033 | OCT loss 0.1803 acc 1.0000
Epoch  8 | DR loss 0.1933 acc 0.7923 | OCT loss 0.1582 acc 1.0000
Epoch  9 | DR loss 0.1741 acc 0.8388 | OCT loss 0.1734 acc 1.0000
✅ Focal loss retraining complete.


In [15]:
from sklearn.metrics import classification_report

# Load epoch 9 (the best)
model.load_state_dict(torch.load("/kaggle/working/joint_epoch9.pth", map_location=device))
model.eval()

# ----- DR Test -----
correct_dr, all_preds_dr, all_labels_dr = 0, [], []
for img, lbl in test_loader:
    img, lbl = img.to(device), lbl.to(device)
    with torch.no_grad():
        out = model(img, task='dr')
        pred = out.argmax(1)
        correct_dr += (pred == lbl).sum().item()
        all_preds_dr.extend(pred.cpu().numpy())
        all_labels_dr.extend(lbl.cpu().numpy())
dr_acc = correct_dr / len(test_loader.dataset)
print(f"Epoch 9 (focal loss) DR Test Accuracy: {dr_acc:.4f}")
print(classification_report(all_labels_dr, all_preds_dr, target_names=['No DR','Mild','Moderate','Severe','Proliferative']))

# ----- OCT Test -----
correct_oct, all_preds_oct, all_labels_oct = 0, [], []
for img, lbl in oct_test_loader:
    img, lbl = img.to(device), lbl.to(device)
    with torch.no_grad():
        out = model(img, task='oct')
        pred = out.argmax(1)
        correct_oct += (pred == lbl).sum().item()
        all_preds_oct.extend(pred.cpu().numpy())
        all_labels_oct.extend(lbl.cpu().numpy())
oct_acc = correct_oct / len(oct_test_loader.dataset)
print(f"Epoch 9 OCT Test Accuracy: {oct_acc:.4f}")
print(classification_report(all_labels_oct, all_preds_oct, target_names=['CNV','DME','DRUSEN','NORMAL']))

Epoch 9 (focal loss) DR Test Accuracy: 0.7787
               precision    recall  f1-score   support

        No DR       0.99      0.96      0.97       199
         Mild       0.44      0.80      0.57        30
     Moderate       0.76      0.54      0.63        87
       Severe       0.25      0.53      0.34        17
Proliferative       0.67      0.42      0.52        33

     accuracy                           0.78       366
    macro avg       0.62      0.65      0.61       366
 weighted avg       0.83      0.78      0.79       366

Epoch 9 OCT Test Accuracy: 0.9907
              precision    recall  f1-score   support

         CNV       0.98      1.00      0.99       242
         DME       1.00      0.99      1.00       242
      DRUSEN       0.99      0.98      0.99       242
      NORMAL       1.00      0.99      0.99       242

    accuracy                           0.99       968
   macro avg       0.99      0.99      0.99       968
weighted avg       0.99      0.99      0.9

In [27]:
model.load_state_dict(torch.load("/kaggle/working/joint_epoch9.pth", map_location=device))
model.to(device)
model.eval()

print("=== DIAGNOSTIC ===")

# 1. Check output shape for DR task
dummy = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    out_dr  = model(dummy, task='dr')
    out_oct = model(dummy, task='oct')

print(f"DR  head output shape : {out_dr.shape}")   # must be [1, 5]
print(f"OCT head output shape : {out_oct.shape}")  # must be [1, 4]

# 2. Check class_names
print(f"class_names ({len(class_names)}) : {class_names}")

# 3. Check model is in eval mode
print(f"model.training        : {model.training}")  # must be False

# 4. Print all head parameters
for name, param in model.named_parameters():
    if 'head' in name.lower() or 'classifier' in name.lower():
        print(f"  {name}: {param.shape}")

=== DIAGNOSTIC ===
DR  head output shape : torch.Size([1, 5])
OCT head output shape : torch.Size([1, 4])
class_names (5) : ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
model.training        : False
  head_dr.weight: torch.Size([5, 768])
  head_dr.bias: torch.Size([5])
  head_oct.weight: torch.Size([4, 768])
  head_oct.bias: torch.Size([4])


In [28]:
import torch, cv2, numpy as np
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Step 1: correct class definitions ──────────────────────────────
DR_CLASSES  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
OCT_CLASSES = ['Normal', 'CNV', 'DME', 'Drusen']

# ── Step 2: load with strict shape check ───────────────────────────
model.load_state_dict(
    torch.load("/kaggle/working/joint_epoch9.pth", map_location=device))
model.to(device)
model.eval()                          # ← disables dropout + fixes batchnorm

# ── Step 3: verify heads before any inference ──────────────────────
dummy = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    out_dr  = model(dummy, task='dr')
    out_oct = model(dummy, task='oct')

assert out_dr.shape[-1]  == len(DR_CLASSES),  \
    f"DR head outputs {out_dr.shape[-1]} classes, expected {len(DR_CLASSES)}"
assert out_oct.shape[-1] == len(OCT_CLASSES), \
    f"OCT head outputs {out_oct.shape[-1]} classes, expected {len(OCT_CLASSES)}"
print("Head shapes verified")

# ── Step 4: fixed deterministic preprocessing ──────────────────────
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
torch.manual_seed(42)

infer_transform = A.Compose([
    A.Resize(224, 224),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ── Step 5: single stable predict function ─────────────────────────
def predict(img_path: str, task: str = 'dr') -> dict:
    assert task in ('dr', 'oct'), "task must be 'dr' or 'oct'"
    classes = DR_CLASSES if task == 'dr' else OCT_CLASSES

    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read: {img_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    tensor = infer_transform(image=img)['image'].unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(tensor, task=task)         # [1, 5] or [1, 4]
        probs  = F.softmax(logits, dim=-1)[0]     # [5] or [4]

    pred_idx   = probs.argmax().item()
    confidence = probs[pred_idx].item()

    assert 0 <= pred_idx < len(classes), \
        f"pred_idx={pred_idx} out of range for {len(classes)} classes"

    print(f"Task       : {task.upper()}")
    print(f"Prediction : {pred_idx} — {classes[pred_idx]}")
    print(f"Confidence : {confidence*100:.2f}%")
    print(f"All probs  :")
    for i, (cls, p) in enumerate(zip(classes, probs.tolist())):
        bar = '█' * int(p * 25)
        mark = ' ← predicted' if i == pred_idx else ''
        print(f"  {i} {cls:20s} {bar:25s} {p*100:.1f}%{mark}")

    return {
        'task':          task,
        'pred_idx':      pred_idx,
        'class_name':    classes[pred_idx],
        'confidence':    round(confidence * 100, 2),
        'probabilities': dict(zip(classes, [round(p*100,2) for p in probs.tolist()]))
    }

# ── Step 6: stability test — must print same result 10 times ───────
print("\n=== STABILITY TEST ===")
results = []
for run in range(1, 11):
    r = predict(sample_img_path, task='dr')
    results.append(r['class_name'])
    print(f"Run {run:2d}: {r['class_name']} ({r['confidence']:.2f}%)")

unique = set(results)
if len(unique) == 1:
    print(f"\nSTABLE — all 10 runs: {results[0]}")
else:
    print(f"\nUNSTABLE — {len(unique)} different results: {unique}")

Head shapes verified

=== STABILITY TEST ===
Task       : DR
Prediction : 0 — No DR
Confidence : 96.97%
All probs  :
  0 No DR                ████████████████████████  97.0% ← predicted
  1 Mild                                           2.7%
  2 Moderate                                       0.3%
  3 Severe                                         0.0%
  4 Proliferative DR                               0.0%
Run  1: No DR (96.97%)
Task       : DR
Prediction : 0 — No DR
Confidence : 97.37%
All probs  :
  0 No DR                ████████████████████████  97.4% ← predicted
  1 Mild                                           2.4%
  2 Moderate                                       0.2%
  3 Severe                                         0.0%
  4 Proliferative DR                               0.0%
Run  2: No DR (97.37%)
Task       : DR
Prediction : 0 — No DR
Confidence : 96.78%
All probs  :
  0 No DR                ████████████████████████  96.8% ← predicted
  1 Mild                              

In [29]:
dummy = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    out_dr  = model(dummy, task='dr')
    out_oct = model(dummy, task='oct')

print(f"DR  head shape : {out_dr.shape}")    # MUST be [1, 5]
print(f"OCT head shape : {out_oct.shape}")   # MUST be [1, 4]
print(f"class_names    : {class_names}")
print(f"model.training : {model.training}")  # MUST be False

DR  head shape : torch.Size([1, 5])
OCT head shape : torch.Size([1, 4])
class_names    : ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
model.training : False
